In [ ]:
# Generate different chord intensity profiles to test Abel inversion algorithm
import numpy as np
import matplotlib.pyplot as plt

# Parameters
num_chords = 10
max_val = 800
min_val = 5

# Create different intensity profiles (one-sided, assuming symmetry)
# Note: chord 0 = outermost, chord 9 = innermost (center)
profiles = {}

# 1. Linear decreasing (what you already have) - edge bright, center dim (hollow plasma)
profiles['Linear Decreasing'] = np.linspace(max_val, min_val, num_chords)

profiles['Linear Increasing'] = np.linspace(min_val, max_val, num_chords)

# 2. Exponential decay - edge bright, rapid falloff toward center
x = np.linspace(0, 3, num_chords)  # 0 to 3 for good decay
profiles['Exponential Decay'] = max_val * np.exp(-x) + min_val

# 3. Power law decay - edge bright, different falloff shape
x_power = np.linspace(0.1, 2, num_chords)  # avoid x=0
profiles['Power Law Decay'] = max_val * x_power**(-2) / x_power[0]**(-2) * 0.8 + min_val

# 4. Square root decay - edge bright, moderate falloff
x_sqrt = np.linspace(0, 4, num_chords)
profiles['Square Root Decay'] = max_val * (1 - np.sqrt(x_sqrt) / np.sqrt(4)) + min_val

# 5. Quadratic decay - edge bright, U-shape (slow then fast falloff)
x_quad = np.linspace(0, 1, num_chords)
profiles['Quadratic Decay'] = min_val + (max_val - min_val) * (1 - x_quad)**2

# 6. Step function - edge/core transition
step_profile = np.full(num_chords, min_val)
step_profile[num_chords//2:] = max_val
profiles['Step Function'] = step_profile

# 7. Flat (constant) - uniform plasma
profiles['Constant'] = np.full(num_chords, (max_val + min_val) / 2)

# 8. Hyperbolic decay - edge bright, very slow falloff 
x_hyp = np.linspace(1, 5, num_chords)
profiles['Hyperbolic Decay'] = max_val / x_hyp * x_hyp[0] * 0.8 + min_val

# 9. Gaussian decay - edge bright, bell-curve-like falloff
x_gauss = np.linspace(-2, 2, num_chords)
profiles['Gaussian Decay'] = max_val * np.exp(-x_gauss**2 / 2) + min_val

# Plot all profiles
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, (name, profile) in enumerate(profiles.items()):
    ax = axes[i]
    ax.plot(range(num_chords), profile, '-o', linewidth=2, markersize=6)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Chord Index')
    ax.set_ylabel('Intensity')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([min_val * 0.8, max_val * 1.2])

plt.tight_layout()
plt.show()

# Print the profile values for easy copying
print("Generated Chord Intensity Profiles:")
print("=" * 50)
for name, profile in profiles.items():
    print(f"\n{name}:")
    print(f"  Values: {profile}")
    print(f"  Min: {np.min(profile):.1f}, Max: {np.max(profile):.1f}, Range: {np.max(profile) - np.min(profile):.1f}")

In [ ]:
%load_ext autoreload
%autoreload 2

from shot_processing import *

# b_raw = np.linspace(-11.8, 11.8, 20)
b_raw = np.linspace(11.8, 0, 10)
I_left = chord_measured[:10]
I_right= np.flip(chord_measured[10:])
I_avg = 0.5 * (I_left + I_right)

# # I_avg = linear
# # print(I_avg)

elec_rad = 7.0

# I_avg = np.full_like(I_left, 10000)
# I_avg[4] = 2000

# Create a synthetic emissivity profile to test the Abel inversion
# This should have N_shells elements (5 in this case since we're using inner_chord_num=4)
N_shells = 5  # This matches what prepare_and_peel will create

# Use one of the profile shapes but adapt it to N_shells elements
profile_name = 'Linear Increasing'  # Change this to test different profiles
original_profile = profiles[profile_name]

# Resample the profile to match N_shells
eps_synthetic = np.interp(np.linspace(0, 1, N_shells), 
                         np.linspace(0, 1, len(original_profile)), 
                         original_profile)

print(f"Synthetic emissivity profile ({profile_name}): {eps_synthetic}")

eps_recovered, (L, edges, b_abs, I_synthetic, edges) = prepare_and_peel(
    b_raw, I_left, lam=1e-3, w_inner=2e4, 
    R=12.42, elec_rad=elec_rad, N_shells=N_shells, inner_chord_num=0, 
    use_tikhonov=True, use_synthetic=True, eps_input=eps_synthetic
)

print("Original emissivity:", eps_synthetic)
print("Recovered emissivity:", eps_recovered)
print("Recovery error:", eps_recovered - eps_synthetic)
print("Synthetic chord intensities:", I_synthetic)
print("Condition number:", np.linalg.cond(L))

In [ ]:
# Create comparison visualization with colormaps for original vs recovered emissivity
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.patches import Circle

# Get variables from prepare_and_peel output
edges_arr = edges
elec = elec_rad 
Rmax = 12.42
b_arr = b_abs
colormap = cm.viridis

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))

# Find common colormap scale for both original and recovered
all_eps = np.concatenate([eps_synthetic, eps_recovered])
eps_norm_common = colors.Normalize(vmin=all_eps.min(), vmax=all_eps.max())

# Left subplot: Original synthetic emissivity
ax1.set_aspect('equal', 'box')
legend_elements_orig = []
for i, (r_outer, r_inner) in enumerate(zip(edges_arr[:-1], edges_arr[1:])):
    if i < len(eps_synthetic):
        color = colormap(eps_norm_common(eps_synthetic[i]))
        
        # Outer circle
        outer_circle = Circle((0.0, 0.0), r_outer, facecolor=color, alpha=0.7, linewidth=1, edgecolor='black')
        ax1.add_patch(outer_circle)
        
        # Inner circle
        if i < len(edges_arr) - 2:
            inner_circle = Circle((0.0, 0.0), r_inner, facecolor='white', alpha=1.0, linewidth=1, edgecolor='black')
            ax1.add_patch(inner_circle)
        
        legend_elements_orig.append(plt.Line2D([0], [0], marker='o', color='w', 
                                             markerfacecolor=color, markersize=10,
                                             label=f'Shell {i}: ε={eps_synthetic[i]:.1e}'))

# Add outermost edge circle and electrode
ax1.add_patch(Circle((0.0, 0.0), edges_arr[0], edgecolor='black', facecolor='none', linewidth=1))
ax1.add_patch(Circle((0.0, 0.0), elec, edgecolor='k', facecolor='grey', alpha=0.8, linewidth=1))

# Add chord lines
for bi in b_arr:
    if bi < elec:
        y_intersect = np.sqrt(elec**2 - bi**2)
        ax1.plot([bi, bi], [Rmax, y_intersect], color='C1', linestyle='--', linewidth=1)
        ax1.plot([-bi, -bi], [Rmax, y_intersect], color='C1', linestyle='--', linewidth=1)
    else:
        ax1.axvline(bi, color='C1', linestyle='--', linewidth=1)
        ax1.axvline(-bi, color='C1', linestyle='--', linewidth=1)

ax1.set_xlim(-Rmax, Rmax)
ax1.set_ylim(-Rmax, Rmax)
ax1.set_xlabel('x (impact parameter)')
ax1.set_ylabel('y')
ax1.set_title(f'Original Synthetic Emissivity')

# Add legend for original emissivity
if legend_elements_orig:
    ax1.legend(handles=legend_elements_orig, loc='upper left', bbox_to_anchor=(0.02, 0.98), fontsize=8)

# Middle subplot: Recovered emissivity  
ax2.set_aspect('equal', 'box')
legend_elements_rec = []
for i, (r_outer, r_inner) in enumerate(zip(edges_arr[:-1], edges_arr[1:])):
    if i < len(eps_recovered):
        color = colormap(eps_norm_common(eps_recovered[i]))
        
        # Outer circle
        outer_circle = Circle((0.0, 0.0), r_outer, facecolor=color, alpha=0.7, linewidth=1, edgecolor='black')
        ax2.add_patch(outer_circle)
        
        # Inner circle
        if i < len(edges_arr) - 2:
            inner_circle = Circle((0.0, 0.0), r_inner, facecolor='white', alpha=1.0, linewidth=1, edgecolor='black')
            ax2.add_patch(inner_circle)
        
        legend_elements_rec.append(plt.Line2D([0], [0], marker='o', color='w', 
                                            markerfacecolor=color, markersize=10,
                                            label=f'Shell {i}: ε={eps_recovered[i]:.1e}'))

# Add outermost edge circle and electrode
ax2.add_patch(Circle((0.0, 0.0), edges_arr[0], edgecolor='black', facecolor='none', linewidth=1))
ax2.add_patch(Circle((0.0, 0.0), elec, edgecolor='k', facecolor='grey', alpha=0.8, linewidth=1))

# Add chord lines
for bi in b_arr:
    if bi < elec:
        y_intersect = np.sqrt(elec**2 - bi**2)
        ax2.plot([bi, bi], [Rmax, y_intersect], color='C1', linestyle='--', linewidth=1)
        ax2.plot([-bi, -bi], [Rmax, y_intersect], color='C1', linestyle='--', linewidth=1)
    else:
        ax2.axvline(bi, color='C1', linestyle='--', linewidth=1)
        ax2.axvline(-bi, color='C1', linestyle='--', linewidth=1)

ax2.set_xlim(-Rmax, Rmax)
ax2.set_ylim(-Rmax, Rmax)
ax2.set_xlabel('x (impact parameter)')
ax2.set_ylabel('y')
ax2.set_title(f'Recovered from Abel Inversion')

# Add legend for recovered emissivity
if legend_elements_rec:
    ax2.legend(handles=legend_elements_rec, loc='upper left', bbox_to_anchor=(0.02, 0.98), fontsize=8)

# Right subplot: Chord intensities (line plot)
indices = np.arange(len(I_synthetic))
ax3.plot(indices, I_synthetic, '-o', color='steelblue', markersize=8, linewidth=2)
ax3.set_xlabel('Chord Index')
ax3.set_ylabel('Synthetic Chord Intensity')
ax3.set_title('Synthetic Chord Intensities')
ax3.set_ylim([0, np.max(I_synthetic)*1.3])
ax3.grid(True, alpha=0.3)

# Add value labels next to each point
max_intensity = np.max(I_synthetic)
for i, intensity in enumerate(I_synthetic):
    ax3.text(i, intensity + max_intensity*0.01, f'{intensity:.0f}', 
             ha='center', va='bottom', fontsize=9)

# Add common colorbar
sm_common = cm.ScalarMappable(cmap=colormap, norm=eps_norm_common)
sm_common.set_array(all_eps)
cbar_common = plt.colorbar(sm_common, ax=[ax1, ax2], shrink=0.8, aspect=20)
cbar_common.set_label('Emissivity', rotation=270, labelpad=15)

# plt.tight_layout()
plt.show()

# Print detailed comparison
print("Detailed Comparison:")
print("=" * 60)
print(f"{'Shell':<6} {'Original':<12} {'Recovered':<12} {'Error':<12} {'Error %':<10}")
print("-" * 60)
for i in range(len(eps_synthetic)):
    orig = eps_synthetic[i]
    rec = eps_recovered[i]
    err = rec - orig
    err_pct = 100 * err / orig if orig != 0 else float('inf')
    print(f"{i:<6} {orig:<12.1f} {rec:<12.1f} {err:<12.1f} {err_pct:<10.1f}")